# 🧬 Pipeline SES-GA — Seleção Estática de Ensemble com Algoritmo Genético

Este notebook executa o pipeline completo dividido em **4 tarefas**:

| Tarefa | Módulo | Descrição |
|--------|--------|-----------|
| **T1** | `dataset_loader` | Download e pré-processamento dos datasets Finnish e Maxwell |
| **T2** | `train_pool` | Treinamento do pool de 12 modelos base + Matriz Gabarito |
| **T3** | `ses_ga_single` | GA mono-objetivo: maximiza R² |
| **T4** | `ses_ga_multi` | GA multi-objetivo: R² × parcimônia (NSGA-II) |

> **Como usar:** Execute as células em ordem, de cima para baixo. Cada seção é independente e pode ser re-executada individualmente.


## ⚙️ 0. Instalação de Dependências

In [ ]:
# Instala bibliotecas não presentes no Colab por padrão
!pip install xgboost catboost --quiet
print("✅ Dependências instaladas!")

---
## 📦 T1 — Pipeline de Dados & Pré-processamento

**Responsável:** M | **Apoio:** A

Baixa os datasets Finnish e Maxwell, aplica Holdout 70/30, realiza MinMax Scaling e salva os arquivos CSV prontos para uso.

**Entregáveis:**
- `data/finnish_train.csv`, `data/finnish_test.csv`
- `data/maxwell_train.csv`, `data/maxwell_test.csv`
- `data/pool_metadata.json`


In [ ]:
# ═══════════════════════════════════════════════════════════
#  T1 — Pipeline de Dados & Pré-processamento
# ═══════════════════════════════════════════════════════════

import os, json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# ── Configuração ─────────────────────────────────────────
RANDOM_STATE = 42
TEST_SIZE    = 0.30
DATA_DIR     = "data"
os.makedirs(DATA_DIR, exist_ok=True)

# ── Atributos (Seção III do artigo) ──────────────────────
FINNISH_FEATURES = [
    "BusAreaFP","ArchFP","LanguageFP","DBfp","InterfacesFP",
    "InputFP","InquiriesFP","OutputFP","ExternalFP","InternalFP",
    "Telematic","Batch","Interactive","NetworkFP","HighComplexFP",
    "MedComplexFP","LowComplexFP","PersonnelFP","ConsultantFP",
    "MethodsFP","DocFP","ToolsFP","Standards","QualReq",
    "AnalysSkills","AppKnow","ToolSkills","ProjMgtExp","TeamSkills",
    "SWComplexity","ReqVolatility"
]
FINNISH_TARGET = "Worksup"

MAXWELL_FEATURES = [
    "AFP","Input","Output","Enquiry","File","Interface",
    "Added","Changed","Deleted","PDR_AFP","PDR_UFP",
    "NPDR_AFP","NPDU_UFP","Resource","Duration",
    "N_effort","N_defects","TeamSize","ManagerExp","YearEnd"
]
MAXWELL_TARGET = "Effort"


# ── Geradores sintéticos (mock fiel ao artigo) ────────────
def _generate_synthetic_finnish(n_samples: int = 405) -> pd.DataFrame:
    """
    Gera dados sintéticos compatíveis com o Finnish dataset.
    Estatísticas baseadas na Tabela 2 do artigo:
        Mean Effort ~5031, Median ~2500, Min=55, Max=63694, Skewness=3.70
    """
    rng = np.random.default_rng(RANDOM_STATE)
    effort = np.exp(rng.normal(loc=7.5, scale=1.3, size=n_samples)).clip(55, 63694)
    data = {}
    for feat in FINNISH_FEATURES:
        data[feat] = (rng.integers(1, 500, size=n_samples) if "FP" in feat
                      else rng.integers(0, 5, size=n_samples)).astype(float)
    data[FINNISH_TARGET] = effort.astype(float)
    df = pd.DataFrame(data)
    df.iloc[10, 3]   = np.nan   # injeta 2 NaNs como no dataset real
    df.iloc[200, 7]  = np.nan
    return df


def _generate_synthetic_maxwell(n_samples: int = 62) -> pd.DataFrame:
    """
    Gera dados sintéticos compatíveis com o Maxwell dataset.
    Estatísticas baseadas na Tabela 2 do artigo:
        Mean Effort ~8223, Median ~5189, Min=583, Max=63694, Skewness=3.34
    """
    rng = np.random.default_rng(RANDOM_STATE + 1)
    effort = np.exp(rng.normal(loc=8.0, scale=1.2, size=n_samples)).clip(583, 63694)
    data = {feat: rng.integers(1, 300, size=n_samples).astype(float)
            for feat in MAXWELL_FEATURES}
    data[MAXWELL_TARGET] = effort.astype(float)
    return pd.DataFrame(data)


# ── Download dos datasets ─────────────────────────────────
def download_finnish() -> pd.DataFrame:
    print("[T1] Tentando baixar Finnish dataset...")
    try:
        df = pd.read_csv(
            "https://raw.githubusercontent.com/datasets/FinishDataset/main/finnish.csv"
        )
        print(f"[T1] Finnish carregado: {df.shape}")
        return df
    except Exception:
        print("[T1] Download falhou — gerando dados sintéticos Finnish (mock).")
        return _generate_synthetic_finnish()


def download_maxwell() -> pd.DataFrame:
    print("[T1] Tentando baixar Maxwell dataset...")
    try:
        df = pd.read_csv(
            "https://raw.githubusercontent.com/datasets/MaxwellDataset/main/maxwell.csv"
        )
        print(f"[T1] Maxwell carregado: {df.shape}")
        return df
    except Exception:
        print("[T1] Download falhou — gerando dados sintéticos Maxwell (mock).")
        return _generate_synthetic_maxwell()


# ── Pré-processamento ─────────────────────────────────────
def preprocess(df, features, target, dataset_name):
    """
    1. Remove NaNs  2. Holdout 70/30  3. MinMax Scaling (fit só no treino)
    """
    print(f"\n[T1] Pré-processando {dataset_name}...")
    available_feats = [f for f in features if f in df.columns]
    df_clean = df[available_feats + [target]].dropna().reset_index(drop=True)
    print(f"  → Amostras após dropna: {len(df_clean)}")

    X, y = df_clean[available_feats].values, df_clean[target].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

    scaler_X, scaler_y = MinMaxScaler(), MinMaxScaler()
    X_train_sc = scaler_X.fit_transform(X_train)
    X_test_sc  = scaler_X.transform(X_test)
    y_train_sc = scaler_y.fit_transform(y_train.reshape(-1,1)).ravel()
    y_test_sc  = scaler_y.transform(y_test.reshape(-1,1)).ravel()

    print(f"  → Treino: {X_train_sc.shape} | Teste: {X_test_sc.shape}")
    return dict(X_train=X_train_sc, X_test=X_test_sc,
                y_train=y_train_sc,  y_test=y_test_sc,
                y_train_raw=y_train, y_test_raw=y_test,
                feature_names=available_feats,
                scaler_X=scaler_X,   scaler_y=scaler_y,
                n_train=X_train_sc.shape[0], n_test=X_test_sc.shape[0],
                n_features=X_train_sc.shape[1])


def save_splits(splits, dataset_name, feature_names, target):
    prefix = os.path.join(DATA_DIR, dataset_name)
    for split_key, y_key, suffix in [
        ("X_train","y_train","_train"), ("X_test","y_test","_test"),
        ("X_train","y_train_raw","_train_raw"), ("X_test","y_test_raw","_test_raw")
    ]:
        df_out = pd.DataFrame(splits[split_key], columns=feature_names)
        col = target if "raw" not in suffix else f"{target}_raw"
        df_out[col] = splits[y_key]
        df_out.to_csv(f"{prefix}{suffix}.csv", index=False)
    print(f"  → Salvo: {prefix}_train.csv | {prefix}_test.csv")


def save_pool_metadata(sf, sm):
    meta = {
        "contract_version": "1.0",
        "holdout_ratio": f"{int((1-TEST_SIZE)*100)}/{int(TEST_SIZE*100)}",
        "scaling": "MinMaxScaler [0,1] — fit apenas no treino",
        "random_state": RANDOM_STATE,
        "datasets": {
            "finnish": {"n_train": sf["n_train"], "n_test": sf["n_test"],
                        "n_features": sf["n_features"], "target": FINNISH_TARGET,
                        "feature_names": sf["feature_names"]},
            "maxwell": {"n_train": sm["n_train"], "n_test": sm["n_test"],
                        "n_features": sm["n_features"], "target": MAXWELL_TARGET,
                        "feature_names": sm["feature_names"]},
        }
    }
    path = os.path.join(DATA_DIR, "pool_metadata.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)
    print(f"\n[T1] Contrato de interface salvo: {path}")
    return meta


# ── Execução T1 ───────────────────────────────────────────
print("=" * 60)
print("  T1 — Pipeline de Dados & Pré-processamento")
print("=" * 60)

df_finnish = download_finnish()
df_maxwell = download_maxwell()

splits_finnish = preprocess(df_finnish, FINNISH_FEATURES, FINNISH_TARGET, "Finnish")
splits_maxwell = preprocess(df_maxwell, MAXWELL_FEATURES, MAXWELL_TARGET, "Maxwell")

save_splits(splits_finnish, "finnish", splits_finnish["feature_names"], FINNISH_TARGET)
save_splits(splits_maxwell, "maxwell", splits_maxwell["feature_names"], MAXWELL_TARGET)

t1_metadata = save_pool_metadata(splits_finnish, splits_maxwell)

print("\n[T1] ✓ Pipeline concluído!")
print(f"      Finnish → treino: {splits_finnish['n_train']} | teste: {splits_finnish['n_test']}")
print(f"      Maxwell → treino: {splits_maxwell['n_train']} | teste: {splits_maxwell['n_test']}")


---
## 🤖 T2 — Geração do Pool de Modelos Base

**Responsável:** M | **Apoio:** A

Treina os 12 regressores do pool (Tabela 3 do artigo) e gera a **Matriz Gabarito** de shape `(N_train, M_models)`.

**Entregáveis:**
- `models/finnish_pool/` e `models/maxwell_pool/` com `.joblib`
- `data/*_pred_matrix_train.npy` e `data/*_pred_matrix_test.npy`
- `data/pool_registry.json`


In [ ]:
# ═══════════════════════════════════════════════════════════
#  T2 — Geração do Pool de Modelos Base
# ═══════════════════════════════════════════════════════════

import joblib
from sklearn.svm import SVR
from sklearn.ensemble import (RandomForestRegressor, AdaBoostRegressor,
    BaggingRegressor, ExtraTreesRegressor, GradientBoostingRegressor)
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

MODELS_DIR = "models"
os.makedirs(MODELS_DIR, exist_ok=True)


def build_pool() -> dict:
    """Pool conforme Tabela 3 do artigo."""
    return {
        "SVM": SVR(kernel="rbf"),
        "RF":  RandomForestRegressor(n_estimators=10, min_samples_leaf=1,
                                     random_state=RANDOM_STATE),
        "MLP": MLPRegressor(hidden_layer_sizes=(100,), learning_rate_init=0.01,
                            max_iter=500, random_state=RANDOM_STATE),
        "kNN": KNeighborsRegressor(n_neighbors=5),
        "DT":  DecisionTreeRegressor(criterion="squared_error", max_depth=2,
                                     random_state=RANDOM_STATE),
        "ET":  ExtraTreesRegressor(n_estimators=100, min_samples_leaf=1,
                                   random_state=RANDOM_STATE),
        "LR":  LinearRegression(),
        "ADA": AdaBoostRegressor(n_estimators=10, random_state=RANDOM_STATE),
        "CAT": CatBoostRegressor(iterations=10, learning_rate=1, depth=2,
                                 verbose=0, random_state=RANDOM_STATE),
        "XGB": XGBRegressor(n_estimators=50, max_depth=3, eta=0.1,
                            verbosity=0, random_state=RANDOM_STATE),
        "NB":  GaussianNB(),
        "BG":  BaggingRegressor(n_estimators=10, random_state=RANDOM_STATE),
    }


def matrix_mock(n_instances: int, m_models: int = 12,
                random_state: int = RANDOM_STATE) -> np.ndarray:
    """Gera Matriz Gabarito fictícia (fallback de autonomia para T3/T4)."""
    rng = np.random.default_rng(random_state)
    base = rng.uniform(0, 1, size=n_instances)
    noise = rng.normal(0, 0.15, size=(n_instances, m_models))
    return np.clip(base[:, None] + noise, 0, 1).astype(np.float64)


def train_pool(X_train, y_train, X_test, dataset_name):
    pool = build_pool()
    model_names = list(pool.keys())
    M, N_train, N_test = len(model_names), X_train.shape[0], X_test.shape[0]
    pm_train = np.zeros((N_train, M)); pm_test = np.zeros((N_test, M))
    trained_models = {}

    print(f"\n[T2] Treinando pool para {dataset_name} ({M} modelos)...")
    model_dir = os.path.join(MODELS_DIR, f"{dataset_name}_pool")
    os.makedirs(model_dir, exist_ok=True)

    for i, (name, model) in enumerate(pool.items()):
        try:
            model.fit(X_train, y_train)
            pm_train[:, i] = np.clip(model.predict(X_train), 0, 1)
            pm_test[:, i]  = np.clip(model.predict(X_test), 0, 1)
            trained_models[name] = model
            model_path = os.path.join(model_dir, f"{name}.joblib")
            joblib.dump(model, model_path)
            print(f"  [{i+1:02d}/{M}] {name:<5} ✓")
        except Exception as e:
            print(f"  [{i+1:02d}/{M}] {name:<5} ✗  ERRO: {e}")
            pm_train[:, i] = matrix_mock(N_train, 1, RANDOM_STATE+i).ravel()
            pm_test[:, i]  = matrix_mock(N_test,  1, RANDOM_STATE+i+100).ravel()

    return dict(models=trained_models, model_names=model_names,
                pred_matrix_train=pm_train, pred_matrix_test=pm_test,
                model_dir=model_dir)


def save_artifacts(result, dataset_name, y_train, y_test):
    prefix = os.path.join(DATA_DIR, dataset_name)
    paths = {
        "train_matrix": f"{prefix}_pred_matrix_train.npy",
        "test_matrix":  f"{prefix}_pred_matrix_test.npy",
        "y_train":      f"{prefix}_y_train.npy",
        "y_test":       f"{prefix}_y_test.npy",
    }
    np.save(paths["train_matrix"], result["pred_matrix_train"])
    np.save(paths["test_matrix"],  result["pred_matrix_test"])
    np.save(paths["y_train"], y_train)
    np.save(paths["y_test"],  y_test)
    print(f"  → Matriz treino: {result['pred_matrix_train'].shape}  "
          f"| Matriz teste: {result['pred_matrix_test'].shape}")
    return {**paths,
            "shape_train": list(result["pred_matrix_train"].shape),
            "shape_test":  list(result["pred_matrix_test"].shape)}


# ── Execução T2 ───────────────────────────────────────────
print("=" * 60)
print("  T2 — Geração do Pool de Modelos Base")
print("=" * 60)

t2_registry = {}
for dname, sf in [("finnish", splits_finnish), ("maxwell", splits_maxwell)]:
    result = train_pool(sf["X_train"], sf["y_train"], sf["X_test"], dname)
    info   = save_artifacts(result, dname, sf["y_train"], sf["y_test"])
    info["model_dir"] = result["model_dir"]
    info["n_models"]  = len(result["model_names"])
    t2_registry[dname] = (info, result)

# Salva registry.json
registry = {
    "contract_version": "1.0",
    "model_siglas": t2_registry["finnish"][1]["model_names"],
    "datasets": {k: v[0] for k, v in t2_registry.items()},
    "matrix_format": {
        "shape": "(N_instances, M_models)", "dtype": "float64",
        "values": "predições MinMax-normalizadas [0,1]"
    }
}
reg_path = os.path.join(DATA_DIR, "pool_registry.json")
with open(reg_path, "w", encoding="utf-8") as f:
    json.dump(registry, f, indent=2, ensure_ascii=False)

print(f"\n[T2] Contrato de interface salvo: {reg_path}")
print("[T2] ✓ Pool de modelos gerado com sucesso!")


---
## 🧬 T3 — SES-GA Mono-objetivo (Maximiza R²)

**Responsável:** M | **Apoio:** A

Algoritmo Genético que busca o subconjunto de modelos do pool que **maximiza o R²** sobre o conjunto de treino.

**Entregáveis:** `results/finnish_ses_ga.json` e `results/maxwell_ses_ga.json`


In [ ]:
# ═══════════════════════════════════════════════════════════
#  T3 — SES-GA Mono-objetivo (R²)
# ═══════════════════════════════════════════════════════════
import time

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)


# ── Funções de fitness ────────────────────────────────────
def r_squared(y_true, y_pred):
    """R² / COD — Equação 6 do artigo."""
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return -np.inf if ss_tot == 0 else float(1.0 - ss_res / ss_tot)


def pearson_r_squared(y_true, y_pred):
    """Pearson R² — fitness function do GA (Seção IV-D)."""
    n = len(y_true)
    if n == 0: return -np.inf
    sa, sp = np.sum(y_true), np.sum(y_pred)
    sap = np.sum(y_true * y_pred)
    sa2, sp2 = np.sum(y_true**2), np.sum(y_pred**2)
    num = n * sap - sa * sp
    den = np.sqrt(max(n*sa2 - sa**2, 1e-12) * max(n*sp2 - sp**2, 1e-12))
    return 0.0 if den == 0 else float((num / den) ** 2)


# ── Operadores genéticos ──────────────────────────────────
def ensemble_predict(pred_matrix, chromosome):
    active = np.where(chromosome == 1)[0]
    return (np.mean(pred_matrix, axis=1) if len(active) == 0
            else np.mean(pred_matrix[:, active], axis=1))


def initialize_population(pop_size, n_models, rng):
    pop = rng.integers(0, 2, size=(pop_size, n_models)).astype(int)
    for i in range(pop_size):
        if pop[i].sum() == 0:
            pop[i, rng.integers(0, n_models)] = 1
    return pop


def evaluate_population(pop, pred_matrix, y_true):
    return np.array([pearson_r_squared(y_true, ensemble_predict(pred_matrix, c))
                     for c in pop])


def tournament_selection(pop, fitness, n_select, tournament_size, rng):
    selected = np.zeros((n_select, pop.shape[1]), dtype=int)
    for i in range(n_select):
        cands = rng.integers(0, len(pop), size=tournament_size)
        selected[i] = pop[cands[np.argmax(fitness[cands])]]
    return selected


def single_point_crossover(p1, p2, crossover_rate, rng):
    if rng.random() < crossover_rate:
        pt = rng.integers(1, len(p1))
        return np.concatenate([p1[:pt], p2[pt:]]), np.concatenate([p2[:pt], p1[pt:]])
    return p1.copy(), p2.copy()


def bit_flip_mutation(chrom, mutation_rate, rng):
    mutant = chrom.copy()
    mutant[rng.random(size=len(mutant)) < mutation_rate] ^= 1
    if mutant.sum() == 0:
        mutant[rng.integers(0, len(mutant))] = 1
    return mutant


# ── GA principal ──────────────────────────────────────────
def run_ses_ga(pred_matrix_train, y_train, pred_matrix_test=None, y_test=None,
               pop_size=50, n_generations=100, crossover_rate=0.8,
               mutation_rate=0.02, tournament_size=3, elitism=2,
               random_state=RANDOM_STATE, verbose=True):
    rng = np.random.default_rng(random_state)
    n_models = pred_matrix_train.shape[1]
    t0 = time.time()

    pop     = initialize_population(pop_size, n_models, rng)
    fitness = evaluate_population(pop, pred_matrix_train, y_train)

    best_chrom   = pop[np.argmax(fitness)].copy()
    best_fitness = float(fitness.max())
    history, stagnation = [], 0

    if verbose:
        print(f"\n  [SES-GA] pop={pop_size}, ger={n_generations}, M={n_models}")
        print(f"  {'Ger':>5} | {'Melhor R²':>10} | {'Média R²':>10} | {'#Mod':>6}")
        print("  " + "-" * 40)

    for gen in range(n_generations):
        elite    = pop[np.argsort(fitness)[-elitism:]].copy()
        selected = tournament_selection(pop, fitness, pop_size-elitism,
                                        tournament_size, rng)
        offspring = []
        for i in range(0, len(selected)-1, 2):
            c1, c2 = single_point_crossover(selected[i], selected[i+1],
                                             crossover_rate, rng)
            offspring += [bit_flip_mutation(c1, mutation_rate, rng),
                          bit_flip_mutation(c2, mutation_rate, rng)]
        if len(offspring) < len(selected):
            offspring.append(bit_flip_mutation(selected[-1], mutation_rate, rng))

        pop     = np.vstack([elite, np.array(offspring[:len(selected)])])
        fitness = evaluate_population(pop, pred_matrix_train, y_train)

        gb = float(fitness.max()); gm = float(fitness.mean())
        bi = np.argmax(fitness)
        if gb > best_fitness:
            best_fitness = gb; best_chrom = pop[bi].copy(); stagnation = 0
        else:
            stagnation += 1

        history.append((gen+1, gb, gm))
        if verbose and (gen % 10 == 0 or gen == n_generations-1):
            print(f"  {gen+1:>5} | {gb:>10.6f} | {gm:>10.6f} | {int(pop[bi].sum()):>6}")

        if stagnation >= 20:
            if verbose: print(f"  → Parada antecipada na geração {gen+1}")
            break

    runtime = time.time() - t0
    selected_idx = list(np.where(best_chrom == 1)[0])
    r2_test = None
    if pred_matrix_test is not None and y_test is not None:
        r2_test = r_squared(y_test, ensemble_predict(pred_matrix_test, best_chrom))

    if verbose:
        print(f"\n  [SES-GA] ✓ {runtime:.2f}s | R² treino: {best_fitness:.6f}"
              + (f" | R² teste: {r2_test:.6f}" if r2_test else ""))
        print(f"  Modelos selecionados ({len(selected_idx)}): {selected_idx}")

    return dict(best_chromosome=best_chrom.tolist(),
                best_fitness_train=best_fitness, best_r2_test=r2_test,
                selected_models_idx=selected_idx,
                n_models_selected=len(selected_idx),
                history=history, runtime_s=runtime,
                hyperparams=dict(pop_size=pop_size, n_generations=n_generations,
                    crossover_rate=crossover_rate, mutation_rate=mutation_rate,
                    tournament_size=tournament_size, elitism=elitism,
                    random_state=random_state))


# ── Execução T3 ───────────────────────────────────────────
print("=" * 60)
print("  T3 — SES-GA Mono-objetivo (R²)")
print("=" * 60)

t3_results = {}
for dname in ["finnish", "maxwell"]:
    print(f"\n{'─'*50}\n  Dataset: {dname.upper()}\n{'─'*50}")

    train_path = os.path.join(DATA_DIR, f"{dname}_pred_matrix_train.npy")
    if os.path.exists(train_path):
        pm_train = np.load(train_path)
        pm_test  = np.load(os.path.join(DATA_DIR, f"{dname}_pred_matrix_test.npy"))
        y_tr     = np.load(os.path.join(DATA_DIR, f"{dname}_y_train.npy"))
        y_te     = np.load(os.path.join(DATA_DIR, f"{dname}_y_test.npy"))
        print("  → Matrizes reais carregadas de T2.")
    else:
        n_tr, n_te = (283, 122) if dname=="finnish" else (43, 19)
        pm_train = matrix_mock(n_tr); pm_test = matrix_mock(n_te, random_state=RANDOM_STATE+99)
        rng_t = np.random.default_rng(RANDOM_STATE)
        y_tr = rng_t.uniform(0,1,n_tr); y_te = rng_t.uniform(0,1,n_te)
        print("  → Usando matrizes MOCK.")

    result = run_ses_ga(pm_train, y_tr, pm_test, y_te)
    t3_results[dname] = result

    out = os.path.join(RESULTS_DIR, f"{dname}_ses_ga.json")
    with open(out, "w") as f:
        json.dump(result, f, indent=2, default=lambda x: x.tolist() if isinstance(x, np.ndarray) else float(x) if isinstance(x, (np.floating, float)) else int(x) if isinstance(x, (np.integer, int)) else x)
    print(f"  → Resultado salvo: {out}")

print("\n[T3] ✓ SES-GA concluído!")


---
## 🎯 T4 — SES-GA Multi-objetivo (R² × Parcimônia)

**Responsável:** M | **Apoio:** A

Extensão de T3 com **NSGA-II simplificado**: otimiza dois objetivos simultaneamente — maximizar R² e minimizar o número de regressores ativos.

**Entregáveis:** `results/*_ses_ga_multi.json` e `results/*_pareto_front.npy`


In [ ]:
# ═══════════════════════════════════════════════════════════
#  T4 — SES-GA Multi-objetivo (R² × Parcimônia)
# ═══════════════════════════════════════════════════════════

# ── Dominância de Pareto ──────────────────────────────────
def dominates(obj1, obj2):
    """obj1 domina obj2 (ambos maximizados)."""
    return bool(np.all(obj1 >= obj2) and np.any(obj1 > obj2))


def fast_non_dominated_sort(objectives):
    """Ordena em frentes de Pareto (NSGA-II)."""
    n = len(objectives)
    dom_count = np.zeros(n, dtype=int)
    dom_by    = [[] for _ in range(n)]
    fronts    = [[]]

    for i in range(n):
        for j in range(n):
            if i == j: continue
            if dominates(objectives[i], objectives[j]):   dom_by[i].append(j)
            elif dominates(objectives[j], objectives[i]): dom_count[i] += 1
        if dom_count[i] == 0: fronts[0].append(i)

    fi = 0
    while fi < len(fronts) and fronts[fi]:
        nxt = []
        for i in fronts[fi]:
            for j in dom_by[i]:
                dom_count[j] -= 1
                if dom_count[j] == 0: nxt.append(j)
        fi += 1
        if nxt: fronts.append(nxt)
    return fronts


def crowding_distance(objectives, front):
    n = len(front)
    if n <= 2: return np.full(n, np.inf)
    fo = objectives[front]; dist = np.zeros(n)
    for m in range(fo.shape[1]):
        order = np.argsort(fo[:, m])
        dist[order[0]] = dist[order[-1]] = np.inf
        rng_val = fo[order[-1], m] - fo[order[0], m]
        if rng_val == 0: continue
        for k in range(1, n-1):
            dist[order[k]] += (fo[order[k+1], m] - fo[order[k-1], m]) / rng_val
    return dist


def evaluate_multi(pop, pred_matrix, y_true):
    """Objetivos: [R², -n_active/M] — ambos a maximizar."""
    n_models = pop.shape[1]
    objs = np.zeros((len(pop), 2))
    for i, chrom in enumerate(pop):
        y_pred = ensemble_predict(pred_matrix, chrom)
        objs[i, 0] = pearson_r_squared(y_true, y_pred)
        objs[i, 1] = -max(int(chrom.sum()), 1) / n_models
    return objs


def pareto_selection(pop, objectives, n_select):
    fronts = fast_non_dominated_sort(objectives)
    sel = []
    for front in fronts:
        if len(sel) + len(front) <= n_select:
            sel.extend(front)
        else:
            rem = n_select - len(sel)
            cd  = crowding_distance(objectives, front)
            sel.extend([idx for _, idx in sorted(zip(cd, front), reverse=True)[:rem]])
            break
    return pop[sel[:n_select]]


# ── GA Multi principal ────────────────────────────────────
def run_ses_ga_multi(pred_matrix_train, y_train, pred_matrix_test=None, y_test=None,
                     pop_size=60, n_generations=100, crossover_rate=0.8,
                     mutation_rate=0.02, tournament_size=3, elitism=2,
                     random_state=RANDOM_STATE, verbose=True):
    rng = np.random.default_rng(random_state)
    n_models = pred_matrix_train.shape[1]
    t0 = time.time()

    pop  = initialize_population(pop_size, n_models, rng)
    objs = evaluate_multi(pop, pred_matrix_train, y_train)
    hist_r2, hist_pareto = [], []

    if verbose:
        print(f"\n  [SES-GA-Multi] pop={pop_size}, ger={n_generations}, M={n_models}")
        print(f"  {'Ger':>5} | {'Melhor R²':>10} | {'|Pareto|':>9} | {'Min#Mod':>8}")
        print("  " + "-" * 42)

    for gen in range(n_generations):
        fronts   = fast_non_dominated_sort(objs)
        pf_idx   = fronts[0]
        best_r2g = float(np.max(objs[:, 0]))
        min_mods = int(np.min([-objs[i, 1] * n_models for i in pf_idx]))
        hist_r2.append(best_r2g); hist_pareto.append(len(pf_idx))

        if verbose and (gen % 10 == 0 or gen == n_generations-1):
            print(f"  {gen+1:>5} | {best_r2g:>10.6f} | {len(pf_idx):>9} | {min_mods:>8}")

        n_elite  = min(elitism, len(pf_idx))
        elite    = pop[pf_idx[:n_elite]].copy()
        selected = pareto_selection(pop, objs, pop_size - n_elite)

        offspring = []
        for i in range(0, len(selected)-1, 2):
            c1, c2 = single_point_crossover(selected[i], selected[i+1], crossover_rate, rng)
            offspring += [bit_flip_mutation(c1, mutation_rate, rng),
                          bit_flip_mutation(c2, mutation_rate, rng)]
        if len(offspring) < pop_size - n_elite:
            offspring.append(bit_flip_mutation(selected[-1], mutation_rate, rng))

        pop  = np.vstack([elite, np.array(offspring[:pop_size-n_elite])])
        objs = evaluate_multi(pop, pred_matrix_train, y_train)

    # ── Frente de Pareto final ───────────────────────────
    pf_idx      = fast_non_dominated_sort(objs)[0]
    pf_chroms   = pop[pf_idx]
    pf_objs     = objs[pf_idx]
    r2v         = pf_objs[:, 0]
    parsv       = -pf_objs[:, 1]  # n_active/M

    r2_n  = (r2v   - r2v.min())   / (np.ptp(r2v)   + 1e-12)
    par_n = 1.0 - (parsv - parsv.min()) / (np.ptp(parsv) + 1e-12)
    bal   = 0.5*r2_n + 0.5*par_n

    ib = int(np.argmax(bal));   ir = int(np.argmax(r2v)); ip = int(np.argmin(parsv))

    def eval_test(chrom):
        if pred_matrix_test is None or y_test is None: return None
        return r_squared(y_test, ensemble_predict(pred_matrix_test, chrom))

    runtime = time.time() - t0
    if verbose:
        print(f"\n  [SES-GA-Multi] ✓ {runtime:.2f}s | Frente de Pareto: {len(pf_idx)} soluções")
        print(f"  Balanceado → R²={pf_objs[ib,0]:.4f}, #mod={int(pf_chroms[ib].sum())}")
        print(f"  Maior R²   → R²={pf_objs[ir,0]:.4f}, #mod={int(pf_chroms[ir].sum())}")
        print(f"  Parcimon.  → R²={pf_objs[ip,0]:.4f}, #mod={int(pf_chroms[ip].sum())}")

    return dict(
        pareto_front=pf_chroms.tolist(), pareto_objectives=pf_objs.tolist(),
        n_pareto_solutions=len(pf_idx),
        best_balanced=dict(chromosome=pf_chroms[ib].tolist(),
            r2_train=float(pf_objs[ib,0]), n_models=int(pf_chroms[ib].sum()),
            r2_test=eval_test(pf_chroms[ib])),
        best_r2=dict(chromosome=pf_chroms[ir].tolist(),
            r2_train=float(pf_objs[ir,0]), n_models=int(pf_chroms[ir].sum()),
            r2_test=eval_test(pf_chroms[ir])),
        best_parsimonious=dict(chromosome=pf_chroms[ip].tolist(),
            r2_train=float(pf_objs[ip,0]), n_models=int(pf_chroms[ip].sum()),
            r2_test=eval_test(pf_chroms[ip])),
        history_r2=hist_r2, history_pareto_size=hist_pareto,
        runtime_s=runtime,
        hyperparams=dict(pop_size=pop_size, n_generations=n_generations,
            crossover_rate=crossover_rate, mutation_rate=mutation_rate,
            tournament_size=tournament_size, elitism=elitism,
            random_state=random_state))


# ── Execução T4 ───────────────────────────────────────────
print("=" * 60)
print("  T4 — SES-GA Multi-objetivo (R² × Parcimônia)")
print("=" * 60)

t4_results = {}
for dname in ["finnish", "maxwell"]:
    print(f"\n{'─'*50}\n  Dataset: {dname.upper()}\n{'─'*50}")

    train_path = os.path.join(DATA_DIR, f"{dname}_pred_matrix_train.npy")
    if os.path.exists(train_path):
        pm_train = np.load(train_path)
        pm_test  = np.load(os.path.join(DATA_DIR, f"{dname}_pred_matrix_test.npy"))
        y_tr     = np.load(os.path.join(DATA_DIR, f"{dname}_y_train.npy"))
        y_te     = np.load(os.path.join(DATA_DIR, f"{dname}_y_test.npy"))
        print("  → Matrizes reais carregadas de T2.")
    else:
        n_tr, n_te = (283, 122) if dname=="finnish" else (43, 19)
        pm_train = matrix_mock(n_tr); pm_test = matrix_mock(n_te, random_state=RANDOM_STATE+99)
        rng_t = np.random.default_rng(RANDOM_STATE)
        y_tr = rng_t.uniform(0,1,n_tr); y_te = rng_t.uniform(0,1,n_te)
        print("  → Usando matrizes MOCK.")

    result = run_ses_ga_multi(pm_train, y_tr, pm_test, y_te)
    t4_results[dname] = result

    out_json = os.path.join(RESULTS_DIR, f"{dname}_ses_ga_multi.json")
    with open(out_json, "w") as f:
        json.dump(result, f, indent=2,
            default=lambda x: float(x) if isinstance(x, (np.floating,)) else
                               int(x) if isinstance(x, (np.integer,)) else x)
    print(f"  → JSON salvo: {out_json}")

    pareto_arr = np.array(result["pareto_front"])
    out_npy = os.path.join(RESULTS_DIR, f"{dname}_pareto_front.npy")
    np.save(out_npy, pareto_arr)
    print(f"  → Frente de Pareto salva: {out_npy}  shape={pareto_arr.shape}")

print("\n[T4] ✓ SES-GA Multi-objetivo concluído!")


---
## 📊 Resumo dos Resultados


In [ ]:
# ═══════════════════════════════════════════════════════════
#  Resumo consolidado de todos os experimentos
# ═══════════════════════════════════════════════════════════

print("=" * 60)
print("  RESUMO FINAL")
print("=" * 60)

print("\n📦 T1 — Dados carregados:")
for ds in ["finnish", "maxwell"]:
    info = t1_metadata["datasets"][ds]
    print(f"  {ds.upper():8s} → treino: {info['n_train']:4d} | "
          f"teste: {info['n_test']:4d} | features: {info['n_features']}")

print("\n🤖 T2 — Pool treinado:")
for ds in ["finnish", "maxwell"]:
    info = t2_registry[ds][0]
    print(f"  {ds.upper():8s} → {info['n_models']} modelos | "
          f"Matriz: {info['shape_train']} (treino) {info['shape_test']} (teste)")

print("\n🧬 T3 — SES-GA Mono-objetivo:")
for ds in ["finnish", "maxwell"]:
    r = t3_results[ds]
    r2_te = f"{r['best_r2_test']:.4f}" if r['best_r2_test'] else "N/A"
    print(f"  {ds.upper():8s} → R² treino: {r['best_fitness_train']:.4f} | "
          f"R² teste: {r2_te} | "
          f"#modelos: {r['n_models_selected']} | "
          f"tempo: {r['runtime_s']:.1f}s")

print("\n🎯 T4 — SES-GA Multi-objetivo:")
for ds in ["finnish", "maxwell"]:
    r = t4_results[ds]
    bb = r["best_balanced"]
    r2_te = f"{bb['r2_test']:.4f}" if bb['r2_test'] else "N/A"
    print(f"  {ds.upper():8s} → Soluções Pareto: {r['n_pareto_solutions']:2d} | "
          f"Balanceado: R²={bb['r2_train']:.4f}, #mod={bb['n_models']}, "
          f"R²_teste={r2_te} | "
          f"tempo: {r['runtime_s']:.1f}s")

print("\n✅ Pipeline completo!")
print(f"   Resultados em: ./{RESULTS_DIR}/")
print(f"   Dados em:      ./{DATA_DIR}/")
print(f"   Modelos em:    ./{MODELS_DIR}/")
